# FIFA World Cup 2026 Prediction

## 1. Import Libraries

## 2. Load Dataset

## 3. Data Cleaning

## 4. Exploratory Data Analysis (EDA)

## 5. Feature Engineering

## 6. Model Training

## 7. Tournament Simulation

## 1. Import Libraries

In [25]:
import pandas as pd

## 2. Load Dataset

In [26]:
df = pd.read_csv("../data/results.csv", encoding="latin1")

In [27]:
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,id
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,1
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,2
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,3
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,4
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,5


## 3. Data Cleaning

In [28]:
df.shape

(43281, 9)

In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43281 entries, 0 to 43280
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        43281 non-null  object 
 1   home_team   43281 non-null  object 
 2   away_team   43281 non-null  object 
 3   home_score  43277 non-null  float64
 4   away_score  43277 non-null  float64
 5   tournament  43281 non-null  object 
 6   city        43281 non-null  object 
 7   country     43281 non-null  object 
 8   id          43281 non-null  int64  
dtypes: float64(2), int64(1), object(6)
memory usage: 3.0+ MB


In [30]:
df.isnull().sum()

date          0
home_team     0
away_team     0
home_score    4
away_score    4
tournament    0
city          0
country       0
id            0
dtype: int64

In [31]:
df = df.dropna()

In [32]:
df.shape

(43277, 9)

In [33]:
df["date"] = pd.to_datetime(df["date"])

In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 43277 entries, 0 to 43276
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        43277 non-null  datetime64[ns]
 1   home_team   43277 non-null  object        
 2   away_team   43277 non-null  object        
 3   home_score  43277 non-null  float64       
 4   away_score  43277 non-null  float64       
 5   tournament  43277 non-null  object        
 6   city        43277 non-null  object        
 7   country     43277 non-null  object        
 8   id          43277 non-null  int64         
dtypes: datetime64[ns](1), float64(2), int64(1), object(5)
memory usage: 3.3+ MB


## 4. Exploratory Data Analysis (EDA)

In [35]:
df["tournament"].nunique()

135

In [36]:
df["home_team"].value_counts().head(10)

home_team
Germany           664
Brazil            600
Argentina         580
Mexico            557
England           530
Sweden            514
France            508
Korea Republic    505
Hungary           481
Italy             470
Name: count, dtype: int64

## 5. Feature Engineering

In [37]:
def get_recent_matches(team, date, n):

    matches = df[
        (
            (df["home_team"] == team) |
            (df["away_team"] == team)
        )
        &
        (df["date"] < date)
    ]

    recent_matches = matches.tail(n)

    return recent_matches

In [38]:
def get_avg_goals_scored(team, recent_matches):

    gf = 0

    for index, row in recent_matches.iterrows():

        if row["home_team"] == team:
            gf = gf + row["home_score"]
        else:
            gf = gf + row["away_score"]

    if recent_matches.shape[0] == 0:
        return 0
    
    goal_avg = gf / recent_matches.shape[0]

    return goal_avg

In [39]:
def get_avg_goals_conceded(team, recent_matches):

    ga = 0

    for index, row in recent_matches.iterrows():

        if row["home_team"] == team:
            ga = ga + row["away_score"]
        else:
            ga = ga + row["home_score"]

    if recent_matches.shape[0] == 0:
        return 0
    
    goal_avg = ga / recent_matches.shape[0]

    return goal_avg

In [40]:
def get_win_percentage(team, recent_matches):
    
    wins = 0

    for index, row in recent_matches.iterrows():

        if team == row["home_team"]:
            if row["home_score"] > row["away_score"]:
                wins += 1

        else:
            if row["away_score"] > row["home_score"]:
                wins += 1

    if recent_matches.shape[0] == 0:
        return 0

    win_percentage = wins / recent_matches.shape[0]

    return win_percentage

In [41]:
def get_avg_points_per_match(team, recent_matches):

    if recent_matches.shape[0] == 0:
        return 0

    points = 0

    for index, row in recent_matches.iterrows():

        if row["home_score"] == row["away_score"]:
            points += 1

        elif row["home_team"] == team:

            if row["home_score"] > row["away_score"]:
                points += 3

        else:

            if row["away_score"] > row["home_score"]:
                points += 3

    avg_points_per_match = points / recent_matches.shape[0]

    return avg_points_per_match

In [42]:
def get_avg_goal_difference(team, recent_matches):

    if recent_matches.shape[0] == 0:
        return 0

    gf = 0
    ga = 0

    for index, row in recent_matches.iterrows():

        if row["home_team"] == team:
            gf += row["home_score"]
            ga += row["away_score"]

        else:
            gf += row["away_score"]
            ga += row["home_score"]

    gd = gf - ga

    avg_goal_difference = gd / recent_matches.shape[0]

    return avg_goal_difference

In [43]:
def get_draw_percentage(team, recent_matches):

    if recent_matches.shape[0] == 0:
        return 0
    
    draws = 0

    for index, row in recent_matches.iterrows():

        if row["home_score"] == row["away_score"]:
            draws+=1

    draw_percentage = draws / recent_matches.shape[0]

    return draw_percentage

In [44]:
def get_loss_percentage(team, recent_matches):

    if recent_matches.shape[0]==0:
        return 0
    
    losses = 0

    for index, row in recent_matches.iterrows():
        if team == row["home_team"]:
            if row["home_score"]<row["away_score"]:
                losses+=1
        else:
            if row["home_score"]>row["away_score"]:
                losses+=1

    lose_percentage = losses/recent_matches.shape[0]

    return lose_percentage
        

6. Dataset Creation for Machine Learning

In [45]:
def create_feature_row(row, home_recent_matches, away_recent_matches):

    # Extract match information
    home_team = row["home_team"]
    away_team = row["away_team"]

    home_score = row["home_score"]
    away_score = row["away_score"]


    # Home team features
    home_avg_gf = get_avg_goals_scored(home_team, home_recent_matches)
    home_avg_ga = get_avg_goals_conceded(home_team, home_recent_matches)
    home_avg_gd = get_avg_goal_difference(home_team, home_recent_matches)

    home_win_percentage = get_win_percentage(home_team, home_recent_matches)
    home_draw_percentage = get_draw_percentage(home_team, home_recent_matches)
    home_loss_percentage = get_loss_percentage(home_team, home_recent_matches)

    home_avg_points = get_avg_points_per_match(home_team, home_recent_matches)


    # Away team features
    away_avg_gf = get_avg_goals_scored(away_team, away_recent_matches)
    away_avg_ga = get_avg_goals_conceded(away_team, away_recent_matches)
    away_avg_gd = get_avg_goal_difference(away_team, away_recent_matches)

    away_win_percentage = get_win_percentage(away_team, away_recent_matches)
    away_draw_percentage = get_draw_percentage(away_team, away_recent_matches)
    away_loss_percentage = get_loss_percentage(away_team, away_recent_matches)

    away_avg_points = get_avg_points_per_match(away_team, away_recent_matches)


    # Create target variable
    if home_score > away_score:
        result = "Home Win"

    elif home_score < away_score:
        result = "Away Win"

    else:
        result = "Draw"


    # Return one training example
    return {
        "home_avg_gf": home_avg_gf,
        "home_avg_ga": home_avg_ga,
        "home_avg_gd": home_avg_gd,
        "home_win_percentage": home_win_percentage,
        "home_draw_percentage": home_draw_percentage,
        "home_loss_percentage": home_loss_percentage,
        "home_avg_points": home_avg_points,

        "away_avg_gf": away_avg_gf,
        "away_avg_ga": away_avg_ga,
        "away_avg_gd": away_avg_gd,
        "away_win_percentage": away_win_percentage,
        "away_draw_percentage": away_draw_percentage,
        "away_loss_percentage": away_loss_percentage,
        "away_avg_points": away_avg_points,

        "result": result
    }

In [46]:
# Number of previous matches to use for feature calculation
RECENT_MATCHES = 10

In [ ]:
# Number of previous matches to use for feature calculation
RECENT_MATCHES = 10

# List to store all training examples
training_data = []

# Loop through every match
for _, row in df.iterrows():

    # Extract match information
    home_team = row["home_team"]
    away_team = row["away_team"]
    date = row["date"]

    # Get recent matches ONLY ONCE
    home_recent_matches = get_recent_matches(
        home_team,
        date,
        RECENT_MATCHES
    )

    away_recent_matches = get_recent_matches(
        away_team,
        date,
        RECENT_MATCHES
    )

    # Skip if either team doesn't have enough previous matches
    if (
        len(home_recent_matches) < RECENT_MATCHES
        or
        len(away_recent_matches) < RECENT_MATCHES
    ):
        continue

    # Create one feature row
    feature_row = create_feature_row(
        row,
        home_recent_matches,
        away_recent_matches
    )

    # Add it to the dataset
    training_data.append(feature_row)

# Convert list of dictionaries into a DataFrame
training_df = pd.DataFrame(training_data)

In [ ]:
training_df.to_csv("../data/training_df.csv", index=False)

In [ ]:
training_df.shape

In [ ]:
training_df.head()

In [ ]:
training_df.info()

In [ ]:
training_df["result"].value_counts

In [ ]:
training_df["result"].value_counts()

In [ ]:
training_df.describe()


In [11]:
import pandas as pd

training_df = pd.read_csv("../data/training_df.csv")

In [12]:
X = training_df.drop(columns=["result"])
y = training_df["result"]

In [49]:
X.head()

,home_avg_gf,home_avg_ga,home_avg_gd,home_win_percentage,home_draw_percentage,home_loss_percentage,home_avg_points,away_avg_gf,away_avg_ga,away_avg_gd,away_win_percentage,away_draw_percentage,away_loss_percentage,away_avg_points
0,2.0,3.0,-1.0,0.3,0.1,0.6,1.0,4.5,1.3,3.2,0.9,0.0,0.1,2.7
1,4.9,1.5,3.4,0.9,0.0,0.1,2.7,3.1,3.2,-0.1,0.4,0.0,0.6,1.2
2,1.3,3.4,-2.1,0.2,0.0,0.8,0.6,3.2,3.4,-0.2,0.4,0.0,0.6,1.2
3,5.1,1.5,3.6,0.9,0.0,0.1,2.7,1.8,3.3,-1.5,0.3,0.0,0.7,0.9
4,3.4,3.6,-0.2,0.4,0.0,0.6,1.2,1.8,3.6,-1.8,0.3,0.0,0.7,0.9


In [50]:
y.head()

0    Away Win
1    Home Win
2    Home Win
3    Home Win
4    Home Win
Name: result, dtype: object

In [ ]:
%pip install scikit-learn

In [4]:
from sklearn.model_selection import train_test_split

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [14]:
X_train.shape

(33280, 14)

In [54]:
X_test.shape

(8320, 14)

In [55]:
y_train.shape

(33280,)

In [56]:
y_test.shape

(8320,)

In [15]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)

model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [16]:
y_pred = model.predict(X_test)

In [59]:
y_pred[0:10]

array(['Home Win', 'Away Win', 'Draw', 'Home Win', 'Home Win', 'Home Win',
       'Home Win', 'Home Win', 'Home Win', 'Home Win'], dtype=object)

In [60]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.4978


In [61]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[ 922  261 1201]
 [ 447  199 1251]
 [ 666  352 3021]]


# ML 7. Hyperparameter Tuning

In [62]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    Away Win       0.45      0.39      0.42      2384
        Draw       0.25      0.10      0.15      1897
    Home Win       0.55      0.75      0.64      4039

    accuracy                           0.50      8320
   macro avg       0.42      0.41      0.40      8320
weighted avg       0.45      0.50      0.46      8320



In [63]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance)

                 Feature  Importance
2            home_avg_gd    0.111043
9            away_avg_gd    0.108718
1            home_avg_ga    0.101729
8            away_avg_ga    0.099957
0            home_avg_gf    0.099815
7            away_avg_gf    0.098776
6        home_avg_points    0.063875
13       away_avg_points    0.063447
4   home_draw_percentage    0.045965
11  away_draw_percentage    0.045907
5   home_loss_percentage    0.044636
12  away_loss_percentage    0.044366
10   away_win_percentage    0.036018
3    home_win_percentage    0.035748


In [64]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

for n in [10, 50, 100, 200, 500]:
    model = RandomForestClassifier(
        n_estimators=n,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    print(f"{n} trees -> Accuracy: {accuracy:.4f}")

10 trees -> Accuracy: 0.4588
50 trees -> Accuracy: 0.4942
100 trees -> Accuracy: 0.4978
200 trees -> Accuracy: 0.5049
500 trees -> Accuracy: 0.5082


In [65]:
for depth in [2, 5, 10, 20, None]:
    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=depth,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    print(f"max_depth={depth} -> Accuracy: {accuracy:.4f}")

max_depth=2 -> Accuracy: 0.5204
max_depth=5 -> Accuracy: 0.5315
max_depth=10 -> Accuracy: 0.5317
max_depth=20 -> Accuracy: 0.5090
max_depth=None -> Accuracy: 0.4978


In [66]:
model = RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.5334


# ML 8. Hyperparameter Tuning using GridSearchCV

In [2]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Parameters to test
param_grid = {
    "n_estimators": [100, 200, 500],
    "max_depth": [5, 10, 20]
}

# Create Grid Search object
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

In [17]:
# Train all models
grid_search.fit(X_train, y_train)

# Best hyperparameters
print("Best Parameters:")
print(grid_search.best_params_)

# Best cross-validation accuracy
print("\nBest Cross Validation Accuracy:")
print(grid_search.best_score_)

# Best trained model
print("\nBest Model:")
print(grid_search.best_estimator_)

Best Parameters:
{'max_depth': 10, 'n_estimators': 500}

Best Cross Validation Accuracy:
0.5344050480769231

Best Model:
RandomForestClassifier(max_depth=10, n_estimators=500, random_state=42)


In [19]:
from sklearn.metrics import accuracy_score

best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print(f"Final Test Accuracy: {accuracy:.4f}")

Final Test Accuracy: 0.5334


In [20]:
import joblib

joblib.dump(best_model, "../models/best_model.pkl")

print("Model saved successfully.")

Model saved successfully.


# ML 9. Predicting Future Matches

In [70]:
def create_prediction_features(home_team, away_team, date):

    # Convert display names to dataset names
    home_team_data = get_dataset_team_name(home_team)
    away_team_data = get_dataset_team_name(away_team)


    # Get recent matches
    home_recent_matches = get_recent_matches(
        home_team_data,
        date,
        RECENT_MATCHES
    )

    away_recent_matches = get_recent_matches(
        away_team_data,
        date,
        RECENT_MATCHES
    )


    # Check if enough matches exist
    if (
        len(home_recent_matches) < RECENT_MATCHES
        or
        len(away_recent_matches) < RECENT_MATCHES
    ):
        return None


    # Home team features
    home_avg_gf = get_avg_goals_scored(
        home_team_data,
        home_recent_matches
    )

    home_avg_ga = get_avg_goals_conceded(
        home_team_data,
        home_recent_matches
    )

    home_avg_gd = get_avg_goal_difference(
        home_team_data,
        home_recent_matches
    )

    home_win_percentage = get_win_percentage(
        home_team_data,
        home_recent_matches
    )

    home_draw_percentage = get_draw_percentage(
        home_team_data,
        home_recent_matches
    )

    home_loss_percentage = get_loss_percentage(
        home_team_data,
        home_recent_matches
    )

    home_avg_points = get_avg_points_per_match(
        home_team_data,
        home_recent_matches
    )


    # Away team features
    away_avg_gf = get_avg_goals_scored(
        away_team_data,
        away_recent_matches
    )

    away_avg_ga = get_avg_goals_conceded(
        away_team_data,
        away_recent_matches
    )

    away_avg_gd = get_avg_goal_difference(
        away_team_data,
        away_recent_matches
    )

    away_win_percentage = get_win_percentage(
        away_team_data,
        away_recent_matches
    )

    away_draw_percentage = get_draw_percentage(
        away_team_data,
        away_recent_matches
    )

    away_loss_percentage = get_loss_percentage(
        away_team_data,
        away_recent_matches
    )

    away_avg_points = get_avg_points_per_match(
        away_team_data,
        away_recent_matches
    )


    return {
        "home_avg_gf": home_avg_gf,
        "home_avg_ga": home_avg_ga,
        "home_avg_gd": home_avg_gd,
        "home_win_percentage": home_win_percentage,
        "home_draw_percentage": home_draw_percentage,
        "home_loss_percentage": home_loss_percentage,
        "home_avg_points": home_avg_points,

        "away_avg_gf": away_avg_gf,
        "away_avg_ga": away_avg_ga,
        "away_avg_gd": away_avg_gd,
        "away_win_percentage": away_win_percentage,
        "away_draw_percentage": away_draw_percentage,
        "away_loss_percentage": away_loss_percentage,
        "away_avg_points": away_avg_points
    }

In [71]:
import pandas as pd

def predict_match(home_team, away_team, date):

    # Create features
    features = create_prediction_features(
        home_team,
        away_team,
        date
    )

    if features is None:
        print("Not enough previous matches.")
        return


    # Convert dictionary to DataFrame
    features_df = pd.DataFrame([features])


    # Predict result
    prediction = best_model.predict(features_df)[0]


    # Predict probabilities
    probabilities = best_model.predict_proba(features_df)[0]


    # Class names
    classes = best_model.classes_


    print(f"\n{home_team} vs {away_team}\n")

    for cls, prob in zip(classes, probabilities):

        if cls == "Home Win":
            print(f"{home_team} Win : {prob*100:.2f}%")

        elif cls == "Away Win":
            print(f"{away_team} Win : {prob*100:.2f}%")

        else:
            print(f"Draw : {prob*100:.2f}%")

    print()

    if prediction == "Home Win":
        print(f"Predicted Winner: {home_team}")

    elif prediction == "Away Win":
        print(f"Predicted Winner: {away_team}")

    else:
        print("Predicted Result: Draw")

In [72]:
team_name_mapping = {
    "South Korea": "Korea Republic",
    "United States": "USA",
    "Turkey": "Turkiye",
    "Iran": "IR Iran",
    "New Zealand": "Aotearoa New Zealand",
    "Cape Verde": "Cabo Verde",
    "Curacao": "Curcao",
    "Ivory Coast": "Cote d'Ivoire"
}


def get_dataset_team_name(team):

    return team_name_mapping.get(team, team)

In [73]:
predict_match(
    "Germany",
    "France",
    "2026-06-15"
)


Germany vs France

France Win : 61.68%
Draw : 20.04%
Germany Win : 18.28%

Predicted Winner: France


In [74]:
def generate_group_fixtures(group):

    matches = []

    for i in range(len(group)):

        for j in range(i + 1, len(group)):

            matches.append((group[i], group[j]))

    return matches

In [75]:
group_k = [
    "Portugal",
    "DR Congo",
    "Uzbekistan",
    "Colombia"
]

In [76]:
fixtures = generate_group_fixtures(group_k)

print(fixtures)

[('Portugal', 'DR Congo'), ('Portugal', 'Uzbekistan'), ('Portugal', 'Colombia'), ('DR Congo', 'Uzbekistan'), ('DR Congo', 'Colombia'), ('Uzbekistan', 'Colombia')]


In [77]:
groups = {
    "A": [
        "Mexico",
        "South Africa",
        "South Korea",
        "Czechia"
    ],

    "B": [
        "Canada",
        "Bosnia and Herzegovina",
        "Qatar",
        "Switzerland"
    ],

    "C": [
        "Brazil",
        "Morocco",
        "Haiti",
        "Scotland"
    ],

    "D": [
        "United States",
        "Paraguay",
        "Australia",
        "Turkey"
    ],

    "E": [
        "Germany",
        "Curacao",
        "Ivory Coast",
        "Ecuador"
    ],

    "F": [
        "Netherlands",
        "Japan",
        "Sweden",
        "Tunisia"
    ],

    "G": [
        "Belgium",
        "Egypt",
        "Iran",
        "New Zealand"
    ],

    "H": [
        "Spain",
        "Cape Verde",
        "Saudi Arabia",
        "Uruguay"
    ],

    "I": [
        "France",
        "Senegal",
        "Iraq",
        "Norway"
    ],

    "J": [
        "Argentina",
        "Algeria",
        "Austria",
        "Jordan"
    ],

    "K": [
    "Portugal",
    "Congo DR",
    "Uzbekistan",
    "Colombia"
    ],

    "L": [
        "England",
        "Croatia",
        "Ghana",
        "Panama"
    ]
}

In [78]:
all_group_fixtures = {}

for group_name, teams in groups.items():

    all_group_fixtures[group_name] = generate_group_fixtures(teams)

In [79]:
for group_name, fixtures in all_group_fixtures.items():

    print(f"\nGroup {group_name}")

    for home_team, away_team in fixtures:
        print(home_team, "vs", away_team)


Group A
Mexico vs South Africa
Mexico vs South Korea
Mexico vs Czechia
South Africa vs South Korea
South Africa vs Czechia
South Korea vs Czechia

Group B
Canada vs Bosnia and Herzegovina
Canada vs Qatar
Canada vs Switzerland
Bosnia and Herzegovina vs Qatar
Bosnia and Herzegovina vs Switzerland
Qatar vs Switzerland

Group C
Brazil vs Morocco
Brazil vs Haiti
Brazil vs Scotland
Morocco vs Haiti
Morocco vs Scotland
Haiti vs Scotland

Group D
United States vs Paraguay
United States vs Australia
United States vs Turkey
Paraguay vs Australia
Paraguay vs Turkey
Australia vs Turkey

Group E
Germany vs Curacao
Germany vs Ivory Coast
Germany vs Ecuador
Curacao vs Ivory Coast
Curacao vs Ecuador
Ivory Coast vs Ecuador

Group F
Netherlands vs Japan
Netherlands vs Sweden
Netherlands vs Tunisia
Japan vs Sweden
Japan vs Tunisia
Sweden vs Tunisia

Group G
Belgium vs Egypt
Belgium vs Iran
Belgium vs New Zealand
Egypt vs Iran
Egypt vs New Zealand
Iran vs New Zealand

Group H
Spain vs Cape Verde
Spain vs

In [80]:
total_fixtures = sum(
    len(fixtures)
    for fixtures in all_group_fixtures.values()
)

print("\nTotal group-stage fixtures:", total_fixtures)


Total group-stage fixtures: 72


In [81]:
import numpy as np

In [86]:
def simulate_match(home_team, away_team, date):

    # Create features for the match
    features = create_prediction_features(
        home_team,
        away_team,
        date
    )

    # If features cannot be created
    if features is None:
        return None

    # Convert features into a DataFrame
    features_df = pd.DataFrame([features])

    # Get probabilities for each possible result
    probabilities = best_model.predict_proba(features_df)[0]

    # Get the class names
    classes = best_model.classes_

    # Simulate one result using the probabilities
    result = np.random.choice(
        classes,
        p=probabilities
    )

    # Generate a score that matches the result
    home_score, away_score = simulate_score(result)

    return {
        "result": result,
        "home_score": home_score,
        "away_score": away_score
    }

In [90]:
result = simulate_match(
    "Germany",
    "France",
    "2026-06-15"
)

print(result)

{'result': 'Away Win', 'home_score': 0, 'away_score': 1}


In [83]:
def simulate_group(fixtures, date):

    results = []

    for home_team, away_team in fixtures:

        match_result = simulate_match(
            home_team,
            away_team,
            date
        )

        if match_result is None:
            continue

        results.append({
            "home_team": home_team,
            "away_team": away_team,
            "result": match_result["result"],
            "home_score": match_result["home_score"],
            "away_score": match_result["away_score"]
        })

    return results

In [91]:
group_k_results = simulate_group(
    all_group_fixtures["K"],
    "2026-06-11"
)

for match in group_k_results:
    print(
        match["home_team"],
        match["home_score"],
        "-",
        match["away_score"],
        match["away_team"]
    )

Portugal 2 - 2 Congo DR
Portugal 2 - 4 Uzbekistan
Portugal 0 - 0 Colombia
Congo DR 5 - 2 Uzbekistan
Congo DR 5 - 2 Colombia
Uzbekistan 2 - 2 Colombia


In [92]:
for team in ["Portugal", "DR Congo", "Uzbekistan", "Colombia"]:

    team_matches = df[
        (
            (df["home_team"] == team) |
            (df["away_team"] == team)
        )
        &
        (df["date"] < "2026-06-11")
    ]

    print(team, "->", len(team_matches))

Portugal -> 662
DR Congo -> 0
Uzbekistan -> 311
Colombia -> 592


In [93]:
df[
    df["home_team"].str.contains("Congo", case=False, na=False) |
    df["away_team"].str.contains("Congo", case=False, na=False)
][["home_team", "away_team"]].drop_duplicates()

,home_team,away_team
2783,Congo DR,Zambia
2997,Zambia,Congo DR
4530,Congo,Cote d'Ivoire
4532,Congo,Cameroon
5002,Gabon,Congo
...,...,...
41606,Bahrain,Congo DR
42450,Congo,South Sudan
42545,South Sudan,Congo
42587,Mauritania,Congo DR


In [94]:
def create_group_standings(teams, results):

    standings = {}

    # Create starting statistics for every team
    for team in teams:
        standings[team] = {
            "P": 0,
            "W": 0,
            "D": 0,
            "L": 0,
            "GF": 0,
            "GA": 0,
            "GD": 0,
            "Pts": 0
        }

    # Process every match
    for match in results:

        home_team = match["home_team"]
        away_team = match["away_team"]

        result = match["result"]

        home_score = match["home_score"]
        away_score = match["away_score"]

        # Matches played
        standings[home_team]["P"] += 1
        standings[away_team]["P"] += 1

        # Goals scored
        standings[home_team]["GF"] += home_score
        standings[away_team]["GF"] += away_score

        # Goals conceded
        standings[home_team]["GA"] += away_score
        standings[away_team]["GA"] += home_score

        # Match result
        if result == "Home Win":

            standings[home_team]["W"] += 1
            standings[away_team]["L"] += 1
            standings[home_team]["Pts"] += 3

        elif result == "Away Win":

            standings[away_team]["W"] += 1
            standings[home_team]["L"] += 1
            standings[away_team]["Pts"] += 3

        elif result == "Draw":

            standings[home_team]["D"] += 1
            standings[away_team]["D"] += 1

            standings[home_team]["Pts"] += 1
            standings[away_team]["Pts"] += 1

    # Calculate goal difference
    for team in teams:
        standings[team]["GD"] = (
            standings[team]["GF"] -
            standings[team]["GA"]
        )

    return standings

In [95]:
group_k_standings = create_group_standings(
    groups["K"],
    group_k_results
)

group_k_standings

{'Portugal': {'P': 3,
  'W': 0,
  'D': 2,
  'L': 1,
  'GF': 4,
  'GA': 6,
  'GD': -2,
  'Pts': 2},
 'Congo DR': {'P': 3,
  'W': 2,
  'D': 1,
  'L': 0,
  'GF': 12,
  'GA': 6,
  'GD': 6,
  'Pts': 7},
 'Uzbekistan': {'P': 3,
  'W': 1,
  'D': 1,
  'L': 1,
  'GF': 8,
  'GA': 9,
  'GD': -1,
  'Pts': 4},
 'Colombia': {'P': 3,
  'W': 0,
  'D': 2,
  'L': 1,
  'GF': 4,
  'GA': 7,
  'GD': -3,
  'Pts': 2}}

In [96]:
group_k_table = pd.DataFrame(group_k_standings).T

group_k_table

,P,W,D,L,GF,GA,GD,Pts
Portugal,3,0,2,1,4,6,-2,2
Congo DR,3,2,1,0,12,6,6,7
Uzbekistan,3,1,1,1,8,9,-1,4
Colombia,3,0,2,1,4,7,-3,2


In [97]:
group_k_table = group_k_table.sort_values(
    by="Pts",
    ascending=False
)

group_k_table

,P,W,D,L,GF,GA,GD,Pts
Congo DR,3,2,1,0,12,6,6,7
Uzbekistan,3,1,1,1,8,9,-1,4
Portugal,3,0,2,1,4,6,-2,2
Colombia,3,0,2,1,4,7,-3,2


In [98]:
def get_group_table(teams, results):

    standings = create_group_standings(
        teams,
        results
    )

    table = pd.DataFrame(standings).T

    table = table.sort_values(
        by=["Pts", "GD", "GF"],
        ascending=[False, False, False]
    )

    return table

In [99]:
group_k_table = get_group_table(
    groups["K"],
    group_k_results
)

group_k_table

,P,W,D,L,GF,GA,GD,Pts
Congo DR,3,2,1,0,12,6,6,7
Uzbekistan,3,1,1,1,8,9,-1,4
Portugal,3,0,2,1,4,6,-2,2
Colombia,3,0,2,1,4,7,-3,2


# SCORE SIMULATION

In [88]:
def simulate_score(result):

    if result == "Home Win":

        away_score = np.random.randint(0, 3)
        home_score = away_score + np.random.randint(1, 4)

    elif result == "Away Win":

        home_score = np.random.randint(0, 3)
        away_score = home_score + np.random.randint(1, 4)

    else:  # Draw

        home_score = np.random.randint(0, 4)
        away_score = home_score

    return home_score, away_score

In [89]:
print("Home Win:", simulate_score("Home Win"))
print("Away Win:", simulate_score("Away Win"))
print("Draw:", simulate_score("Draw"))

Home Win: (1, 0)
Away Win: (0, 2)
Draw: (1, 1)


# SIMULATE ALL GROUPS

In [100]:
all_group_results = {}
all_group_tables = {}

for group_name, teams in groups.items():

    fixtures = all_group_fixtures[group_name]

    results = simulate_group(
        fixtures,
        "2026-06-11"
    )

    table = get_group_table(
        teams,
        results
    )

    all_group_results[group_name] = results
    all_group_tables[group_name] = table

KeyboardInterrupt: 

In [ ]:
for group_name, table in all_group_tables.items():

    print(f"\nGROUP {group_name}")
    print(table)

In [ ]:
qualified_teams = []

third_placed_teams = []

for group_name, table in all_group_tables.items():

    # Top 2 automatically qualify
    qualified_teams.append({
        "team": table.index[0],
        "group": group_name,
        "position": 1
    })

    qualified_teams.append({
        "team": table.index[1],
        "group": group_name,
        "position": 2
    })

    # Store third-placed team separately
    third_placed_teams.append({
        "team": table.index[2],
        "group": group_name,
        "Pts": table.iloc[2]["Pts"],
        "GD": table.iloc[2]["GD"],
        "GF": table.iloc[2]["GF"]
    })

In [ ]:
third_placed_df = pd.DataFrame(third_placed_teams)

third_placed_df = third_placed_df.sort_values(
    by=["Pts", "GD", "GF"],
    ascending=False
)

third_placed_df

In [ ]:
best_third_placed = third_placed_df.head(8)

best_third_placed

In [ ]:
# Add the 8 best third-placed teams
for _, row in best_third_placed.iterrows():

    qualified_teams.append({
        "team": row["team"],
        "group": row["group"],
        "position": 3
    })


# Convert to DataFrame
qualified_teams_df = pd.DataFrame(qualified_teams)

print("Total qualified teams:", len(qualified_teams_df))

qualified_teams_df

In [ ]:
failed_matches = []

for group_name, fixtures in all_group_fixtures.items():

    for home_team, away_team in fixtures:

        match_result = simulate_match(
            home_team,
            away_team,
            "2026-06-11"
        )

        if match_result is None:
            failed_matches.append({
                "group": group_name,
                "home_team": home_team,
                "away_team": away_team
            })

pd.DataFrame(failed_matches)

In [ ]:
problem_teams = [
    "South Korea",
    "United States",
    "Turkey",
    "Curacao",
    "Ivory Coast",
    "Iran",
    "New Zealand",
    "Cape Verde"
]

for team in problem_teams:

    matches = df[
        df["home_team"].str.contains(
            team,
            case=False,
            na=False
        )
        |
        df["away_team"].str.contains(
            team,
            case=False,
            na=False
        )
    ]

    print(f"\nSearching for: {team}")

    print(
        pd.unique(
            pd.concat([
                matches["home_team"],
                matches["away_team"]
            ])
        )
    )

In [ ]:
search_names = [
    "Korea Republic",
    "USA",
    "Turkiye",
    "Curacao",
    "Curcao",
    "Curaçao",
    "Cote d'Ivoire",
    "Côte d'Ivoire",
    "Cape Verde",
    "Cabo Verde"
]

all_teams = pd.unique(
    pd.concat([
        df["home_team"],
        df["away_team"]
    ])
)

for name in search_names:
    matches = [
        team for team in all_teams
        if name.lower() in team.lower()
    ]

    print(f"\nSearching for: {name}")
    print(matches)

In [ ]:
all_dataset_teams = set(
    pd.concat([
        df["home_team"],
        df["away_team"]
    ]).unique()
)

all_world_cup_teams = []

for group_name, teams in groups.items():
    all_world_cup_teams.extend(teams)

for team in all_world_cup_teams:

    if team in all_dataset_teams:
        print(f"✓ {team}")

    else:
        print(f"❌ {team}")

In [ ]:
team_name_mapping = {
    "South Korea": "Korea Republic",
    "United States": "USA",
    "Turkey": "Turkiye",
    "Curacao": "Curcao",
    "Ivory Coast": "Cote d'Ivoire",
    "Iran": "IR Iran",
    "New Zealand": "Aotearoa New Zealand",
    "Cape Verde": "Cabo Verde"
}


def get_dataset_team_name(team):
    return team_name_mapping.get(team, team)

In [ ]:
for team in all_world_cup_teams:
    
    dataset_name = get_dataset_team_name(team)

    if dataset_name in all_dataset_teams:
        print(f"✓ {team} -> {dataset_name}")
    else:
        print(f"❌ {team} -> {dataset_name}")

In [ ]:
simulate_match(
    "South Korea",
    "Mexico",
    "2026-06-11"
)

In [ ]:
print(qualified_teams)

In [ ]:
def get_qualified_team(group, position):
    
    team = qualified_teams_df[
        (qualified_teams_df["group"] == group) &
        (qualified_teams_df["position"] == position)
    ]
    
    return team.iloc[0]["team"]

In [ ]:
print(get_qualified_team("A", 1))
print(get_qualified_team("C", 2))
print(get_qualified_team("K", 3))

In [ ]:
print(
    qualified_teams_df[
        qualified_teams_df["group"].isin(["A", "C", "K"])
    ]
)

In [ ]:
qualified_teams = []

for group_name, table in all_group_tables.items():

    # Top 2 teams qualify automatically
    for position in range(2):

        qualified_teams.append({
            "team": table.index[position],
            "group": group_name,
            "position": position + 1
        })


# Create DataFrame
qualified_teams_df = pd.DataFrame(qualified_teams)

qualified_teams_df

In [ ]:
for _, row in best_third_teams.iterrows():

    qualified_teams_df.loc[len(qualified_teams_df)] = {
        "team": row["team"],
        "group": row["group"],
        "position": 3
    }


qualified_teams_df

In [ ]:
best_third_teams = pd.DataFrame({
    "team": [
        "Paraguay",
        "Uzbekistan",
        "Iran",
        "Ecuador",
        "Japan",
        "Haiti",
        "Qatar",
        "Algeria"
    ],
    
    "group": [
        "D",
        "K",
        "G",
        "E",
        "F",
        "C",
        "B",
        "J"
    ]
})

best_third_teams

In [ ]:
for _, row in best_third_teams.iterrows():

    qualified_teams_df.loc[len(qualified_teams_df)] = {
        "team": row["team"],
        "group": row["group"],
        "position": 3
    }

qualified_teams_df

In [ ]:
print(get_qualified_team("A", 1))
print(get_qualified_team("C", 2))
print(get_qualified_team("K", 3))

In [ ]:
qualified_teams_data = [
    # Group A
    {"team": "Mexico", "group": "A", "position": 1},
    {"team": "South Africa", "group": "A", "position": 2},

    # Group B
    {"team": "Switzerland", "group": "B", "position": 1},
    {"team": "Bosnia and Herzegovina", "group": "B", "position": 2},

    # Group C
    {"team": "Morocco", "group": "C", "position": 1},
    {"team": "Brazil", "group": "C", "position": 2},

    # Group D
    {"team": "United States", "group": "D", "position": 1},
    {"team": "Turkey", "group": "D", "position": 2},

    # Group E
    {"team": "Germany", "group": "E", "position": 1},
    {"team": "Curacao", "group": "E", "position": 2},

    # Group F
    {"team": "Tunisia", "group": "F", "position": 1},
    {"team": "Netherlands", "group": "F", "position": 2},

    # Group G
    {"team": "Belgium", "group": "G", "position": 1},
    {"team": "Egypt", "group": "G", "position": 2},

    # Group H
    {"team": "Cape Verde", "group": "H", "position": 1},
    {"team": "Spain", "group": "H", "position": 2},

    # Group I
    {"team": "France", "group": "I", "position": 1},
    {"team": "Norway", "group": "I", "position": 2},

    # Group J
    {"team": "Austria", "group": "J", "position": 1},
    {"team": "Argentina", "group": "J", "position": 2},

    # Group K
    {"team": "Portugal", "group": "K", "position": 1},
    {"team": "Congo DR", "group": "K", "position": 2},

    # Group L
    {"team": "England", "group": "L", "position": 1},
    {"team": "Croatia", "group": "L", "position": 2},

    # Best 8 third-place teams
    {"team": "Paraguay", "group": "D", "position": 3},
    {"team": "Uzbekistan", "group": "K", "position": 3},
    {"team": "Iran", "group": "G", "position": 3},
    {"team": "Ecuador", "group": "E", "position": 3},
    {"team": "Japan", "group": "F", "position": 3},
    {"team": "Haiti", "group": "C", "position": 3},
    {"team": "Qatar", "group": "B", "position": 3},
    {"team": "Algeria", "group": "J", "position": 3}
]

qualified_teams_df = pd.DataFrame(qualified_teams_data)

qualified_teams_df

In [ ]:
print(get_qualified_team("A", 1))
print(get_qualified_team("C", 2))
print(get_qualified_team("K", 3))

In [ ]:
qualified_lookup = {}

for _, row in qualified_teams_df.iterrows():

    key = f"{row['group']}{row['position']}"

    qualified_lookup[key] = row["team"]

In [ ]:
qualified_lookup

In [ ]:
round_of_32 = {
    "Match 73": (
        qualified_lookup["A2"],
        qualified_lookup["B2"]
    ),

    "Match 75": (
        qualified_lookup["F1"],
        qualified_lookup["C2"]
    ),

    "Match 76": (
        qualified_lookup["C1"],
        qualified_lookup["F2"]
    ),

    "Match 78": (
        qualified_lookup["E2"],
        qualified_lookup["I2"]
    ),

    "Match 83": (
        qualified_lookup["K2"],
        qualified_lookup["L2"]
    ),

    "Match 84": (
        qualified_lookup["H1"],
        qualified_lookup["J2"]
    ),

    "Match 86": (
        qualified_lookup["J1"],
        qualified_lookup["H2"]
    ),

    "Match 88": (
        qualified_lookup["D2"],
        qualified_lookup["G2"]
    )
}

In [ ]:
third_place_groups = sorted([
    key[0]
    for key in qualified_lookup
    if key.endswith("3")
])

third_place_groups

In [ ]:
import random

third_place_teams = [
    qualified_lookup[key]
    for key in qualified_lookup
    if key.endswith("3")
]

random.shuffle(third_place_teams)

third_place_teams

In [ ]:
# The 8 Round of 32 matches
# where a group winner will face
# one of the best third-place teams

third_place_match_numbers = [
    "Match 74",
    "Match 77",
    "Match 79",
    "Match 80",
    "Match 81",
    "Match 82",
    "Match 85",
    "Match 87"
]


# Group winners that will face
# the third-place teams

third_place_opponents = [
    qualified_lookup["E1"],
    qualified_lookup["I1"],
    qualified_lookup["A1"],
    qualified_lookup["L1"],
    qualified_lookup["D1"],
    qualified_lookup["G1"],
    qualified_lookup["B1"],
    qualified_lookup["K1"]
]


# Assign each shuffled third-place team
# to a Round of 32 match

for match_number, opponent, third_place_team in zip(
    third_place_match_numbers,
    third_place_opponents,
    third_place_teams
):
    
    round_of_32[match_number] = (
        opponent,
        third_place_team
    )


# Display all Round of 32 matches

for match, teams in round_of_32.items():
    print(
        f"{match}: "
        f"{teams[0]} vs {teams[1]}"
    )

In [ ]:
def simulate_knockout_match(
    home_team,
    away_team,
    date
):

    result = simulate_match(
        home_team,
        away_team,
        date
    )

    # Normal win
    if result["home_score"] > result["away_score"]:
        winner = home_team

    # Away win
    elif result["away_score"] > result["home_score"]:
        winner = away_team

    # Draw -> temporary random winner
    else:
        winner = random.choice([
            home_team,
            away_team
        ])

    return {
        "home_team": home_team,
        "away_team": away_team,
        "home_score": result["home_score"],
        "away_score": result["away_score"],
        "winner": winner
    }

In [ ]:
round_of_32_results = {}

for match, teams in round_of_32.items():

    home_team = teams[0]
    away_team = teams[1]

    result = simulate_knockout_match(
        home_team,
        away_team,
        "2026-06-28"
    )

    round_of_32_results[match] = result

    print(
        f"{match}: "
        f"{home_team} {result['home_score']} - "
        f"{result['away_score']} {away_team}"
    )

    print("Winner:", result["winner"])
    print()

In [ ]:
round_of_16 = {
    "Match 89": (
        round_of_32_results["Match 73"]["winner"],
        round_of_32_results["Match 74"]["winner"]
    ),

    "Match 90": (
        round_of_32_results["Match 75"]["winner"],
        round_of_32_results["Match 76"]["winner"]
    ),

    "Match 91": (
        round_of_32_results["Match 77"]["winner"],
        round_of_32_results["Match 78"]["winner"]
    ),

    "Match 92": (
        round_of_32_results["Match 79"]["winner"],
        round_of_32_results["Match 80"]["winner"]
    ),

    "Match 93": (
        round_of_32_results["Match 81"]["winner"],
        round_of_32_results["Match 82"]["winner"]
    ),

    "Match 94": (
        round_of_32_results["Match 83"]["winner"],
        round_of_32_results["Match 84"]["winner"]
    ),

    "Match 95": (
        round_of_32_results["Match 85"]["winner"],
        round_of_32_results["Match 86"]["winner"]
    ),

    "Match 96": (
        round_of_32_results["Match 87"]["winner"],
        round_of_32_results["Match 88"]["winner"]
    )
}

for match, teams in round_of_16.items():
    print(f"{match}: {teams[0]} vs {teams[1]}")

In [ ]:
round_of_16_results = {}

for match, teams in round_of_16.items():

    home_team = teams[0]
    away_team = teams[1]

    result = simulate_knockout_match(
        home_team,
        away_team,
        "2026-07-03"
    )

    round_of_16_results[match] = result

    print(
        f"{match}: "
        f"{home_team} {result['home_score']} - "
        f"{result['away_score']} {away_team}"
    )

    print("Winner:", result["winner"])
    print()

In [ ]:
quarter_finals = {
    "Match 97": (
        round_of_16_results["Match 89"]["winner"],
        round_of_16_results["Match 90"]["winner"]
    ),

    "Match 98": (
        round_of_16_results["Match 91"]["winner"],
        round_of_16_results["Match 92"]["winner"]
    ),

    "Match 99": (
        round_of_16_results["Match 93"]["winner"],
        round_of_16_results["Match 94"]["winner"]
    ),

    "Match 100": (
        round_of_16_results["Match 95"]["winner"],
        round_of_16_results["Match 96"]["winner"]
    )
}


for match, teams in quarter_finals.items():
    print(f"{match}: {teams[0]} vs {teams[1]}")

In [ ]:
quarter_final_results = {}

for match, teams in quarter_finals.items():

    home_team = teams[0]
    away_team = teams[1]

    result = simulate_knockout_match(
        home_team,
        away_team,
        "2026-07-07"
    )

    quarter_final_results[match] = result

    print(
        f"{match}: "
        f"{home_team} {result['home_score']} - "
        f"{result['away_score']} {away_team}"
    )

    print("Winner:", result["winner"])
    print()

In [ ]:
semi_finals = {
    "Match 101": (
        quarter_final_results["Match 97"]["winner"],
        quarter_final_results["Match 98"]["winner"]
    ),

    "Match 102": (
        quarter_final_results["Match 99"]["winner"],
        quarter_final_results["Match 100"]["winner"]
    )
}


for match, teams in semi_finals.items():
    print(f"{match}: {teams[0]} vs {teams[1]}")

In [ ]:
semi_final_results = {}

for match, teams in semi_finals.items():

    home_team = teams[0]
    away_team = teams[1]

    result = simulate_knockout_match(
        home_team,
        away_team,
        "2026-07-14"
    )

    semi_final_results[match] = result

    print(
        f"{match}: "
        f"{home_team} {result['home_score']} - "
        f"{result['away_score']} {away_team}"
    )

    print("Winner:", result["winner"])
    print()

In [ ]:
world_cup_final = {
    "Match 103": (
        semi_final_results["Match 101"]["winner"],
        semi_final_results["Match 102"]["winner"]
    )
}

for match, teams in world_cup_final.items():
    print(f"{match}: {teams[0]} vs {teams[1]}")

In [ ]:
final_results = {}

for match, teams in world_cup_final.items():

    home_team = teams[0]
    away_team = teams[1]

    result = simulate_knockout_match(
        home_team,
        away_team,
        "2026-07-19"
    )

    final_results[match] = result

    print(
        f"{match}: "
        f"{home_team} {result['home_score']} - "
        f"{result['away_score']} {away_team}"
    )

    print("🏆 WORLD CUP WINNER:", result["winner"])

In [ ]:
third_place_match = {
    "Match 104": (
        "France",
        "Belgium"
    )
}

for match, teams in third_place_match.items():
    print(f"{match}: {teams[0]} vs {teams[1]}")

In [ ]:
third_place_results = {}

for match, teams in third_place_match.items():

    home_team = teams[0]
    away_team = teams[1]

    result = simulate_knockout_match(
        home_team,
        away_team,
        "2026-07-18"
    )

    third_place_results[match] = result

    print(
        f"{match}: "
        f"{home_team} {result['home_score']} - "
        f"{result['away_score']} {away_team}"
    )

    print("🥉 THIRD PLACE:", result["winner"])

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import joblib

# Load the already-created training data
training_df = pd.read_csv("../data/training_df.csv")

# Separate features and target
X = training_df.drop(columns=["result"])
y = training_df["result"]

# Same train/test split as V1
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Same GridSearchCV setup as V1
param_grid = {
    "n_estimators": [100, 200, 500],
    "max_depth": [5, 10, 20]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

# Train
grid_search.fit(X_train, y_train)

# Display results
print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross Validation Accuracy:")
print(grid_search.best_score_)

print("\nBest Model:")
print(grid_search.best_estimator_)

# Test accuracy
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print(f"\nFinal Test Accuracy: {accuracy:.4f}")

# Save model using sklearn 1.9.0
joblib.dump(best_model, "../models/best_model.pkl")

print("\nModel saved successfully.")

Best Parameters:
{'max_depth': 10, 'n_estimators': 500}

Best Cross Validation Accuracy:
0.5344050480769231

Best Model:
RandomForestClassifier(max_depth=10, n_estimators=500, random_state=42)

Final Test Accuracy: 0.5334

Model saved successfully.
